In [ ]:
import pandas as pd
import os

folder = "clean_protokols"
all_dfs = []

for file in os.listdir(folder):
    if file.endswith('.xlsx') or file.endswith('.xls'):
        parts = file.replace('.xlsx', '').replace('.xls', '').split('_')
        subject = parts[0]
        year = int(parts[1])

        df = pd.read_excel(os.path.join(folder, file))
        df.columns = ['№', 'Класс', 'Итог', 'Тип диплома']
        df['Предмет'] = subject
        df['Год'] = year
        all_dfs.append(df)

df_all = pd.concat(all_dfs, ignore_index=True)
print(df_all.head())


max_scores = {
    ('математика', 2022): 70,
    ('математика', 2023): 70,
    ('математика', 2024): 70,
    ('математика', 2025): 70,
    ('биология', 2022): 100,
    ('биология', 2023): 100,
    ('биология', 2024): 100,
    ('биология', 2025): 312,
    ('общество', 2022): 100,
    ('общество', 2023): 100,
    ('общество', 2024): 100,
    ('общество', 2025): 200,
    ('литература', 2022): 100,
    ('литература', 2023): 100,
    ('литература', 2024): 100,
    ('литература', 2025): 100,
}
df_all['Итог'] = pd.to_numeric(df_all['Итог'], errors='coerce')
def normalize_score(row):
    key = (row['Предмет'], row['Год'])
    max_score = max_scores.get(key)
    if max_score and pd.notna(row['Итог']):
        return (row['Итог'] / max_score) * 100
    else:
        return None

df_all['Нормализованные баллы'] = df_all.apply(normalize_score, axis=1)

df_all.to_excel('все_данные.xlsx', index=False)
thresholds_df = pd.read_excel("проходные_баллы.xlsx")
df_all['Класс'] = df_all['Класс'].astype('Int64')
thresholds_df['Класс'] = thresholds_df['Класс'].astype('Int64')
df_all = df_all.merge(
    thresholds_df,
    how='left',
    on=['Предмет', 'Год', 'Класс']
)
print(df_all[['Предмет', 'Год', 'Класс', 'Итог', 'Баллы для всероссийского этапа']].head(10))
df_all.to_excel('все_данные_с_порогами.xlsx', index=False)